In [1]:
import cloudscraper
import json
from bs4 import BeautifulSoup
import re
import time
import random
import pandas as pd
import os

In [5]:
# ==========================================
# 1. CẤU HÌNH CƠ BẢN
# ==========================================
scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'mobile': False}
)

DANH_MUC = [
    {"Thành_Phố": "Đà Nẵng", "URL": "https://phongtro123.com/tinh-thanh/da-nang"},
    {"Thành_Phố": "Hồ Chí Minh", "URL": "https://phongtro123.com/tinh-thanh/ho-chi-minh"},
    {"Thành_Phố": "Hà Nội", "URL": "https://phongtro123.com/tinh-thanh/ha-noi"}
]

MAX_PAGES = 50 
FILE_NAME = 'data_phongtro_raw.csv'

all_rooms = []
seen_urls = set() 
df_old = None

# ==========================================
# 2. HỆ THỐNG KHÔI PHỤC TIẾN TRÌNH (RESUME)
# ==========================================
if os.path.exists(FILE_NAME):
    print(f"[*] Phát hiện file dữ liệu cũ '{FILE_NAME}'. Đang nạp dữ liệu để cào tiếp...")
    try:
        df_old = pd.read_csv(FILE_NAME)
        all_rooms = df_old.to_dict('records')
        print(f"[*] Đã khôi phục thành công {len(all_rooms)} phòng cũ!")
    except Exception as e:
        print(f"[!] Lỗi khi đọc file cũ: {e}")

# ==========================================
# 3. HÀM XỬ LÝ SỐ LIỆU CƠ BẢN
# ==========================================
def extract_price(price_str):
    """Hàm làm sạch chuỗi giá về dạng số thực (Triệu VNĐ)"""
    if not price_str: return None
    p = str(price_str).lower().replace(",", ".")
    match = re.search(r'(\d+(?:\.\d+)?)', p)
    if not match: return None
    val = float(match.group(1))
    
    if 'triệu' in p: return val
    if 'đồng' in p or 'vnđ' in p: return val / 1000000
    if 'k' in p: return val / 1000
    return val

# ==========================================
# 4. QUÁ TRÌNH CÀO DỮ LIỆU THÔ
# ==========================================
print("\n🚀 BẮT ĐẦU CÀO DỮ LIỆU PHÒNG TRỌ RAW (ĐÀ NẴNG, HÀ NỘI, TP.HCM)...")
print("⚠️ LƯU Ý: Nhấn Ctrl + C bất cứ lúc nào để DỪNG và LƯU file.")

try:
    for dm in DANH_MUC:
        thanh_pho = dm["Thành_Phố"]
        
        start_page = 1
        if df_old is not None and thanh_pho in df_old['Thành_Phố'].values:
            so_luong_da_cao = len(df_old[df_old['Thành_Phố'] == thanh_pho])
            start_page = (so_luong_da_cao // 20) + 1 
            print(f"\n👉 ĐANG QUÉT: {thanh_pho.upper()} (Chạy tiếp từ Trang {start_page})")
        else:
            print(f"\n👉 ĐANG QUÉT: {thanh_pho.upper()} (Bắt đầu từ Trang 1)")
            
        if start_page > MAX_PAGES:
            print(f"    -> Đã cào đủ {MAX_PAGES} trang cho {thanh_pho}. Bỏ qua!")
            continue
        
        for page in range(start_page, MAX_PAGES + 1):
            list_url = f"{dm['URL']}?page={page}" if page > 1 else dm['URL']
            print(f"  + Trang {page} / {MAX_PAGES}...")
            
            try:
                res = scraper.get(list_url, timeout=15)
                if res.status_code != 200: continue
                
                soup = BeautifulSoup(res.text, "html.parser")
                scripts = soup.find_all('script', type='application/ld+json')
                
                json_rooms = []
                has_new_room = False 
                
                for script in scripts:
                    try:
                        raw_json = script.string.strip() if script.string else ""
                        if not raw_json: continue
                        json_data = json.loads(raw_json)
                        
                        if isinstance(json_data, dict) and json_data.get('@type') in ['Hostel', 'Apartment', 'RealEstateListing']:
                            url_val = json_data.get('url', '')
                            if not url_val: continue
                            
                            code_match = re.search(r'-pr(\d+)\.html', url_val)
                            ma_tin_val = code_match.group(1) if code_match else None
                            
                            if ma_tin_val and ma_tin_val not in seen_urls:
                                seen_urls.add(ma_tin_val)
                                has_new_room = True 
                                
                                room = {
                                    "Mã_Tin": ma_tin_val,
                                    "Thành_Phố": thanh_pho,
                                    "Tiêu_Đề": json_data.get('name', ''),
                                }
                                
                                if 'floorSize' in json_data and isinstance(json_data['floorSize'], dict):
                                    room["Diện_Tích_m2"] = float(json_data['floorSize'].get('value', 0))
                                    
                                if 'priceRange' in json_data:
                                    room["Giá_Cho_Thuê"] = int(json_data.get('priceRange', 0)) / 1000000
                                    
                                address_obj = json_data.get('address', {})
                                if isinstance(address_obj, dict):
                                    full_address = address_obj.get('streetAddress', '')
                                    qh_match = re.search(r'(?:Quận|Huyện)\s+([^,]+)', full_address, re.IGNORECASE)
                                    room["Quận_Huyện"] = qh_match.group(1).strip() if qh_match else "Không rõ"
                                
                                # Lưu URL để request chi tiết
                                room["URL"] = url_val
                                json_rooms.append(room)
                    except Exception:
                        continue
                
                if not has_new_room:
                    print(f"    -> [!] Đã hết tin mới tại {thanh_pho}. Chuyển thành phố!")
                    break
                    
                print(f"    -> Lấy được sơ bộ {len(json_rooms)} phòng. Vào lấy text raw...")
                
                found_rooms = 0
                for room in json_rooms:
                    try:
                        detail_res = scraper.get(room["URL"], timeout=15)
                        detail_soup = BeautifulSoup(detail_res.text, "html.parser")
                        
                        # --- 1. LẤY GIÁ VÀ DIỆN TÍCH (Nếu thiếu) ---
                        if "Giá_Cho_Thuê" not in room or pd.isna(room.get("Giá_Cho_Thuê")) or room["Giá_Cho_Thuê"] == 0:
                            price_tag = detail_soup.select_one(".post-price, .item-price, .text-green")
                            if price_tag:
                                room["Giá_Cho_Thuê"] = extract_price(price_tag.text)
                                
                        if "Diện_Tích_m2" not in room or pd.isna(room.get("Diện_Tích_m2")) or room["Diện_Tích_m2"] == 0:
                            for span in detail_soup.find_all('span'):
                                span_text = span.text.strip().lower()
                                if 'm2' in span_text or 'm²' in span_text:
                                    a_match = re.search(r'(\d+(?:\.\d+)?)\s*m', span_text)
                                    if a_match:
                                        room["Diện_Tích_m2"] = float(a_match.group(1))
                                        break 
                        
                        # --- 2. LẤY THỜI GIAN ĐĂNG ---
                        time_tag = detail_soup.find('time')
                        if time_tag and time_tag.has_attr('title'):
                            room["Thời_Gian_Đăng"] = time_tag['title'].replace("Cập nhật:", "").strip()
                        else:
                            room["Thời_Gian_Đăng"] = "Không rõ"
                        
                        # XỬ LÝ CỘT MÔ TẢ (Chỉ bám vào đúng thẻ h2 hoặc h3)
                        h2_desc = detail_soup.find(lambda tag: tag.name in ['h2', 'h3'] and tag.get_text(strip=True).lower() == 'thông tin mô tả')
                        
                        if h2_desc and h2_desc.parent:
                            # Lấy toàn bộ text của thẻ div cha, ngăn cách các thẻ <p> bằng khoảng trắng
                            raw_desc = h2_desc.parent.get_text(separator=' ', strip=True)
                            # Cắt bỏ luôn chữ "Thông tin mô tả" ở đầu câu
                            raw_desc = re.sub(r'^Thông tin mô tả\s*', '', raw_desc, flags=re.IGNORECASE)
                            # Xóa khoảng trắng thừa
                            room["Mô_Tả"] = re.sub(r'\s+', ' ', raw_desc).strip()
                        else:
                            # Fallback lại cách cũ phòng trường hợp web dùng giao diện khác
                            desc_tag = detail_soup.select_one(".post-content, .section-content")
                            room["Mô_Tả"] = re.sub(r'\s+', ' ', desc_tag.text).strip() if desc_tag else ""
                        
                        noi_bat_text = []
                        noi_bat_items = detail_soup.select(".col-3 .text-body")
                        for item in noi_bat_items:
                            # Bỏ qua các nhãn tiêu đề làm mờ
                            if 'opacity-75' not in item.get('class', []) and item.text:
                                noi_bat_text.append(item.text.strip())
                                
                        # Gộp thành 1 chuỗi ngăn cách bởi dấu phẩy
                        room["Nổi_Bật"] = ", ".join(noi_bat_text) 
                        
                        # Bỏ cột URL đi cho file CSV nhẹ bớt
                        room.pop("URL", None)
                        
                        all_rooms.append(room)
                        found_rooms += 1
                        
                        time.sleep(random.uniform(0.5, 1.2))
                        
                    except Exception as e:
                        continue
                        
                print(f"    -> Đã lấy xong raw text {found_rooms} phòng.")
                
            except Exception as e:
                print(f"  [!] Lỗi kết nối trang: {e}")
                break

except KeyboardInterrupt:
    print("\n[!] ⛔ BẠN VỪA BẤM DỪNG (CTRL+C). Đang tiến hành lưu toàn bộ dữ liệu raw...")

# ==========================================
# 5. XUẤT DỮ LIỆU RAW
# ==========================================
if all_rooms:
    df = pd.DataFrame(all_rooms)
    
    # Sắp xếp đúng thứ tự các cột nguyên bản
    cols = ['Mã_Tin', 'Thời_Gian_Đăng', 'Thành_Phố', 'Quận_Huyện', 'Giá_Cho_Thuê', 
            'Diện_Tích_m2', 'Tiêu_Đề', 'Mô_Tả', 'Nổi_Bật']
            
    df = df[[c for c in cols if c in df.columns]]
    df = df.dropna(subset=['Giá_Cho_Thuê', 'Diện_Tích_m2'])
    df = df.drop_duplicates(subset=['Mã_Tin'], keep='last')
    
    df.to_csv(FILE_NAME, index=False, encoding='utf-8-sig')
    print(f"\n🎉 XUẤT SẮC! Đã lưu an toàn {len(df)} phòng trọ dạng RAW vào file '{FILE_NAME}'")
else:
    print("\n[!] Chưa cào được dữ liệu nào.")


🚀 BẮT ĐẦU CÀO DỮ LIỆU PHÒNG TRỌ RAW (ĐÀ NẴNG, HÀ NỘI, TP.HCM)...
⚠️ LƯU Ý: Nhấn Ctrl + C bất cứ lúc nào để DỪNG và LƯU file.

👉 ĐANG QUÉT: ĐÀ NẴNG (Bắt đầu từ Trang 1)
  + Trang 1 / 50...
    -> Lấy được sơ bộ 19 phòng. Vào lấy text raw...
    -> Đã lấy xong raw text 19 phòng.
  + Trang 2 / 50...
    -> Lấy được sơ bộ 20 phòng. Vào lấy text raw...
    -> Đã lấy xong raw text 20 phòng.
  + Trang 3 / 50...
    -> Lấy được sơ bộ 20 phòng. Vào lấy text raw...
    -> Đã lấy xong raw text 20 phòng.
  + Trang 4 / 50...
    -> Lấy được sơ bộ 20 phòng. Vào lấy text raw...
    -> Đã lấy xong raw text 20 phòng.
  + Trang 5 / 50...
    -> Lấy được sơ bộ 18 phòng. Vào lấy text raw...
    -> Đã lấy xong raw text 18 phòng.
  + Trang 6 / 50...
    -> Lấy được sơ bộ 19 phòng. Vào lấy text raw...
    -> Đã lấy xong raw text 19 phòng.
  + Trang 7 / 50...
    -> Lấy được sơ bộ 20 phòng. Vào lấy text raw...
    -> Đã lấy xong raw text 20 phòng.
  + Trang 8 / 50...
    -> Lấy được sơ bộ 20 phòng. Vào lấy t